In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
path = "../data/cleaned_nse_data.csv"
df = pd.read_csv(path)
df.head()

###  Shape, Data Types & Missing Values

In [ ]:
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"\nDate range: {df['trade_date'].min()} → {df['trade_date'].max()}")
print(f"\n--- Data Types ---")
print(df.dtypes)
print(f"\n--- Missing Values ---")
print(df.isnull().sum())

###  Statistical Summary of Numeric Columns

In [ ]:
df.describe().round(2)

In [ ]:
# Convert trade_date to datetime for proper time-series plotting
df['trade_date'] = pd.to_datetime(df['trade_date'])

### NIFTY 50 Closing Price Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['trade_date'], df['close'], linewidth=0.8, color='steelblue')
ax.set_title('NIFTY 50 — Closing Price (2005–2025)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Close (₹)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Trading Volume Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(df['trade_date'], df['shares_traded'] / 1e6, width=2, color='coral', alpha=0.7)
ax.set_title('Daily Trading Volume (millions of shares)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Shares Traded (M)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Distribution of Daily Returns
Daily return = percentage change in closing price from one day to the next.

In [ ]:
df['daily_return'] = df['close'].pct_change() * 100  # in %

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['daily_return'].dropna(), bins=80, color='mediumseagreen', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_title('Distribution of Daily Returns (%)', fontsize=14)
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Frequency')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean daily return:   {df['daily_return'].mean():.4f}%")
print(f"Std dev:             {df['daily_return'].std():.4f}%")
print(f"Biggest single-day gain:  {df['daily_return'].max():.2f}%")
print(f"Biggest single-day drop:  {df['daily_return'].min():.2f}%")

### Year-wise Summary
Average closing price, total volume, and annual return for each year.

In [ ]:
df['year'] = df['trade_date'].dt.year

yearly = df.groupby('year').agg(
    avg_close=('close', 'mean'),
    total_volume=('shares_traded', 'sum'),
    trading_days=('close', 'count'),
    year_open=('open', 'first'),
    year_close=('close', 'last')
).round(2)

yearly['annual_return_%'] = ((yearly['year_close'] - yearly['year_open']) / yearly['year_open'] * 100).round(2)
yearly

### Annual Return Bar Chart

In [ ]:
colors = ['crimson' if r < 0 else 'seagreen' for r in yearly['annual_return_%']]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(yearly.index, yearly['annual_return_%'], color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('NIFTY 50 — Annual Returns (%)', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Return (%)')
ax.grid(axis='y', alpha=0.3)

for i, (yr, ret) in enumerate(zip(yearly.index, yearly['annual_return_%'])):
    ax.text(yr, ret + (1.5 if ret >= 0 else -3), f"{ret:.1f}%", ha='center', fontsize=8)

plt.tight_layout()
plt.show()

### Box Plot — Closing Price by Year
Shows the spread and outliers of closing prices each year.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
sns.boxplot(data=df, x='year', y='close', palette='viridis', ax=ax, fliersize=2)
ax.set_title('Yearly Distribution of Closing Prices', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Close (₹)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Rolling Moving Averages (20-day & 200-day)
The classic SMA crossover — when the short-term MA crosses above the long-term MA it's often seen as a bullish signal, and vice versa.

In [ ]:
# Compute rolling moving averages
df['sma_20'] = df['close'].rolling(window=20).mean()
df['sma_200'] = df['close'].rolling(window=200).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['trade_date'], df['close'], linewidth=0.5, alpha=0.5, color='gray', label='Close')
ax.plot(df['trade_date'], df['sma_20'], linewidth=1.2, color='dodgerblue', label='20-day SMA')
ax.plot(df['trade_date'], df['sma_200'], linewidth=1.5, color='orangered', label='200-day SMA')
ax.set_title('NIFTY 50 — 20-day vs 200-day Moving Averages', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Price (₹)')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Rolling Volatility (20-day & 60-day)
Standard deviation of daily returns over a rolling window — spikes indicate periods of market turbulence.

In [ ]:
# Rolling standard deviation of daily returns (annualised)
df['roll_vol_20'] = df['daily_return'].rolling(window=20).std() * np.sqrt(252)
df['roll_vol_60'] = df['daily_return'].rolling(window=60).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df['trade_date'], df['roll_vol_20'], linewidth=0.9, color='tomato', alpha=0.7, label='20-day rolling vol')
ax.plot(df['trade_date'], df['roll_vol_60'], linewidth=1.3, color='darkred', label='60-day rolling vol')
ax.set_title('NIFTY 50 — Annualised Rolling Volatility (%)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Volatility (%)')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Rolling Average Volume (20-day)
Smoothed trading volume to spot liquidity trends and unusual activity periods.

In [ ]:
df['roll_vol_avg_20'] = df['shares_traded'].rolling(window=20).mean()

fig, ax = plt.subplots(figsize=(16, 4))
ax.fill_between(df['trade_date'], df['roll_vol_avg_20'] / 1e6, alpha=0.4, color='mediumpurple')
ax.plot(df['trade_date'], df['roll_vol_avg_20'] / 1e6, linewidth=0.8, color='indigo')
ax.set_title('NIFTY 50 — 20-day Rolling Average Volume', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Avg Shares Traded (M)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Rolling Sharpe Ratio (60-day)
Sharpe ratio = mean return / std of returns (annualised). Values above 1 suggest good risk-adjusted performance.

In [ ]:
# Annualised rolling Sharpe (assuming 0% risk-free rate for simplicity)
roll_mean = df['daily_return'].rolling(window=60).mean() * 252
roll_std  = df['daily_return'].rolling(window=60).std() * np.sqrt(252)
df['roll_sharpe_60'] = roll_mean / roll_std

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df['trade_date'], df['roll_sharpe_60'], linewidth=0.9, color='teal')
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.axhline(1, color='green', linewidth=0.7, linestyle=':', alpha=0.7, label='Sharpe = 1')
ax.axhline(-1, color='red', linewidth=0.7, linestyle=':', alpha=0.7, label='Sharpe = -1')
ax.fill_between(df['trade_date'], df['roll_sharpe_60'], 0,
                where=df['roll_sharpe_60'] > 0, color='seagreen', alpha=0.15)
ax.fill_between(df['trade_date'], df['roll_sharpe_60'], 0,
                where=df['roll_sharpe_60'] < 0, color='crimson', alpha=0.15)
ax.set_title('NIFTY 50 — 60-day Rolling Sharpe Ratio (annualised)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Rolling Max Drawdown (200-day)
Drawdown = how far the price has fallen from its recent peak. Deeper drawdowns mean larger losses from the top.

In [ ]:
# Rolling max drawdown over a 200-day window
roll_max = df['close'].rolling(window=200, min_periods=1).max()
drawdown = (df['close'] - roll_max) / roll_max * 100  # in %

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(df['trade_date'], drawdown, 0, color='crimson', alpha=0.4)
ax.plot(df['trade_date'], drawdown, linewidth=0.7, color='darkred')
ax.set_title('NIFTY 50 — Rolling Max Drawdown (200-day window)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Deepest drawdown: {drawdown.min():.2f}%  on  {df.loc[drawdown.idxmin(), 'trade_date'].strftime('%Y-%m-%d')}")

### Full Feature Correlation (with Engineered Features)
Includes derived features like daily return, daily range, and volatility to uncover hidden relationships.

In [ ]:
# Engineer useful features
df['daily_range'] = df['high'] - df['low']                          # intraday price spread
df['daily_range_pct'] = df['daily_range'] / df['close'] * 100       # range as % of close
df['gap'] = df['open'] - df['close'].shift(1)                       # overnight gap
df['gap_pct'] = df['gap'] / df['close'].shift(1) * 100              # gap as %
df['log_return'] = np.log(df['close'] / df['close'].shift(1))       # log return
df['turnover_per_share'] = df['turnover_cr'] / (df['shares_traded'] / 1e7)  # avg trade value

feat_cols = ['open', 'high', 'low', 'close', 'shares_traded', 'turnover_cr',
             'daily_return', 'daily_range', 'daily_range_pct', 'gap_pct', 'log_return']

corr_full = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_full, dtype=bool), k=1)  # upper triangle mask
sns.heatmap(corr_full, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Full Feature Correlation Heatmap', fontsize=15)
plt.tight_layout()
plt.show()

### Lagged Feature Correlation with Next-Day Return
Which of today's features best predict **tomorrow's return**? This is critical for pattern recognition & forecasting.

In [ ]:
# Create lagged features (today's values) vs tomorrow's return
df['next_day_return'] = df['daily_return'].shift(-1)

lag_cols = ['daily_return', 'daily_range_pct', 'gap_pct', 'log_return',
            'shares_traded', 'turnover_cr', 'daily_range']

lag_corr = df[lag_cols + ['next_day_return']].corr()['next_day_return'].drop('next_day_return')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['crimson' if v < 0 else 'steelblue' for v in lag_corr.values]
bars = ax.barh(lag_corr.index, lag_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title("Correlation of Today's Features → Tomorrow's Return", fontsize=14)
ax.set_xlabel('Pearson Correlation')

for bar, val in zip(bars, lag_corr.values):
    ax.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Rolling Correlation: Return vs Volume (60-day)
Does volume move *with* or *against* returns? This relationship changes over time — especially during crashes.

In [ ]:
# 60-day rolling correlation between daily return and volume
roll_corr_ret_vol = df['daily_return'].rolling(window=60).corr(df['shares_traded'])

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df['trade_date'], roll_corr_ret_vol, linewidth=0.8, color='darkorange')
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.fill_between(df['trade_date'], roll_corr_ret_vol, 0,
                where=roll_corr_ret_vol > 0, color='seagreen', alpha=0.2)
ax.fill_between(df['trade_date'], roll_corr_ret_vol, 0,
                where=roll_corr_ret_vol < 0, color='crimson', alpha=0.2)
ax.set_title('60-day Rolling Correlation: Daily Return vs Volume', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Pearson r')
ax.set_ylim(-0.6, 0.6)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Year-wise Correlation Heatmaps (Price vs Volume vs Volatility)
How do correlations between key metrics change across different market eras?

In [ ]:
# Correlation between key metrics for selected market eras
eras = {
    '2005–2007\n(Bull Run)':     (2005, 2007),
    '2008\n(GFC Crash)':         (2008, 2008),
    '2009–2010\n(Recovery)':     (2009, 2010),
    '2015–2016\n(Sideways)':     (2015, 2016),
    '2020\n(COVID)':             (2020, 2020),
    '2021–2023\n(Post-COVID)':   (2021, 2023),
}

key_cols = ['daily_return', 'daily_range_pct', 'shares_traded', 'turnover_cr', 'gap_pct']

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (era_name, (y1, y2)) in enumerate(eras.items()):
    era_df = df[(df['year'] >= y1) & (df['year'] <= y2)]
    era_corr = era_df[key_cols].corr()
    sns.heatmap(era_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, linewidths=0.5, ax=axes[idx],
                cbar=idx == 2, square=True)
    axes[idx].set_title(era_name, fontsize=12, fontweight='bold')

fig.suptitle('How Key Correlations Shift Across Market Eras', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### Decomposition — Trend, Seasonality & Residual
Breaks the closing price into its core components using multiplicative seasonal decomposition.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Use close price indexed by date for decomposition
ts = df.set_index('trade_date')['close']
# Multiplicative decomposition with ~1 year period (252 trading days)
decomp = seasonal_decompose(ts, model='multiplicative', period=252)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
decomp.observed.plot(ax=axes[0], color='steelblue', linewidth=0.6)
axes[0].set_ylabel('Observed')
axes[0].set_title('Time Series Decomposition (Multiplicative, period=252 trading days)', fontsize=14)

decomp.trend.plot(ax=axes[1], color='orangered', linewidth=1.2)
axes[1].set_ylabel('Trend')

decomp.seasonal.plot(ax=axes[2], color='seagreen', linewidth=0.5)
axes[2].set_ylabel('Seasonality')

decomp.resid.plot(ax=axes[3], color='gray', linewidth=0.5)
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Date')

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Bollinger Bands (20-day, 2σ)
Price envelope around the 20-day SMA. Touches to the upper/lower band signal overbought/oversold conditions.

In [ ]:
window = 20
df['bb_mid'] = df['close'].rolling(window).mean()
df['bb_std'] = df['close'].rolling(window).std()
df['bb_upper'] = df['bb_mid'] + 2 * df['bb_std']
df['bb_lower'] = df['bb_mid'] - 2 * df['bb_std']

# Plot last 2 years for clarity
recent = df[df['trade_date'] >= '2024-01-01'].copy()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(recent['trade_date'], recent['close'], linewidth=0.9, color='black', label='Close')
ax.plot(recent['trade_date'], recent['bb_mid'], linewidth=1, color='dodgerblue', label='20-day SMA')
ax.fill_between(recent['trade_date'], recent['bb_upper'], recent['bb_lower'],
                alpha=0.15, color='dodgerblue', label='Bollinger Band (2σ)')
ax.plot(recent['trade_date'], recent['bb_upper'], linewidth=0.6, color='dodgerblue', linestyle='--')
ax.plot(recent['trade_date'], recent['bb_lower'], linewidth=0.6, color='dodgerblue', linestyle='--')
ax.set_title('NIFTY 50 — Bollinger Bands (2024–2025)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Price (₹)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Cumulative Return (log scale)
Growth of ₹1 invested at the start — log scale reveals whether growth is exponential or decelerating.

In [ ]:
# Cumulative return — growth of ₹1
df['cum_return'] = (1 + df['daily_return'] / 100).cumprod()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df['trade_date'], df['cum_return'], linewidth=1, color='teal')
ax.set_yscale('log')
ax.set_title('NIFTY 50 — Cumulative Return (₹1 invested in Jan 2005)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Growth of ₹1 (log scale)')
ax.grid(alpha=0.3, which='both')

final_val = df['cum_return'].iloc[-1]
ax.annotate(f'₹{final_val:.2f}', xy=(df['trade_date'].iloc[-1], final_val),
            fontsize=12, fontweight='bold', color='teal',
            xytext=(-80, 15), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='teal'))
plt.tight_layout()
plt.show()

print(f"₹1 invested in Jan 2005 → ₹{final_val:.2f} by Dec 2025  ({(final_val-1)*100:.0f}% total return)")

### Monthly Seasonality — Average Return by Calendar Month
Is there a "best month" to be in the market? Classic pattern recognition across 21 years.

In [ ]:
df['month'] = df['trade_date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly_ret = df.groupby('month')['daily_return'].mean()
monthly_std = df.groupby('month')['daily_return'].std()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Average return per month
colors_m = ['crimson' if v < 0 else 'seagreen' for v in monthly_ret.values]
axes[0].bar(month_names, monthly_ret.values, color=colors_m, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.7)
axes[0].set_title('Average Daily Return by Month (%)', fontsize=13)
axes[0].set_ylabel('Mean Daily Return (%)')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(monthly_ret.values):
    axes[0].text(i, v + (0.003 if v >= 0 else -0.008), f'{v:.3f}', ha='center', fontsize=9)

# Volatility per month
axes[1].bar(month_names, monthly_std.values, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('Daily Return Volatility by Month (%)', fontsize=13)
axes[1].set_ylabel('Std Dev (%)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()